In [3]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras.datasets import california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

(X_train_full, y_train_full), (X_test, y_test) = california_housing.load_data(version="large", test_split=0.2, seed=42)

print(X_train_full.shape)
print(y_train_full.shape)
print(X_test.shape)
print(y_test.shape)

(16512, 8)
(16512,)
(4128, 8)
(4128,)


In [4]:
X_train_full

array([[-119.01  ,   36.06  ,   25.    , ..., 1392.    ,  359.    ,
           1.6812],
       [-119.46  ,   35.14  ,   30.    , ..., 1565.    ,  584.    ,
           2.5313],
       [-122.44  ,   37.8   ,   52.    , ..., 1310.    ,  963.    ,
           3.4801],
       ...,
       [-120.71  ,   38.34  ,   16.    , ...,  559.    ,  213.    ,
           4.4531],
       [-117.13  ,   32.91  ,   16.    , ..., 1619.    ,  584.    ,
           4.    ],
       [-117.93  ,   33.71  ,   10.    , ..., 1581.    ,  633.    ,
           4.1366]], shape=(16512, 8), dtype=float32)

In [ ]:
y_train_full

array([ 47700.,  45800., 500001., ..., 144300., 154700., 158800.],
      dtype=float32)

In [6]:
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_valid_scaled = scaler.transform(X_valid)
X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape)
print(X_valid_scaled.shape)
print(X_test_scaled.shape)

(13209, 8)
(3303, 8)
(4128, 8)


In [7]:
X_train_scaled

array([[-1.3428202 ,  1.1067797 , -1.8047258 , ..., -0.985674  ,
        -1.0625787 ,  0.71047664],
       [-1.2779112 ,  0.9663889 , -0.6124807 , ...,  0.67205924,
         1.3191351 , -0.6548123 ],
       [ 0.8240624 , -0.94761074, -0.85092974, ..., -0.8894823 ,
        -0.54527146,  0.7625645 ],
       ...,
       [-0.88847244,  1.3922421 , -0.13558266, ...,  0.12097979,
         0.6967909 , -0.5779382 ],
       [-0.96835923,  0.6481688 , -0.7714467 , ..., -0.24490818,
        -0.34307528,  0.07447005],
       [-0.45909023,  0.76984036, -1.4073107 , ...,  0.26391885,
        -0.3220679 , -1.0104141 ]], shape=(13209, 8), dtype=float32)

In [ ]:
model = keras.models.Sequential([
    keras.layers.Dense(30, activation="relu", input_shape=X_train_scaled.shape[1:]),
    keras.layers.Dense(30, activation="relu"),
    keras.layers.Dense(1)
])

model.compile(loss="mse", optimizer="adam")

history = model.fit(X_train_scaled, y_train, epochs=20, validation_data=(X_valid_scaled, y_valid))

In [9]:
mse_test = model.evaluate(X_test_scaled, y_test)
y_pred = model.predict(X_test_scaled)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

print(f"Test Set Loss (MSE): {mse_test}")
print(f"Mean Squared Error (MSE): {mse}")
print(f"Root Mean Squared Error (RMSE): {rmse}")
print(f"Mean Absolute Error (MAE): {mae}")

# Small explanation of metrics:
# MSE (Mean Squared Error): Measures the average squared difference. Penalizes larger errors heavily. Smaller is better.
# RMSE (Root Mean Squared Error): Error in the original unit of the target. Easier to interpret than MSE. Smaller is better.
# MAE (Mean Absolute Error): Measures the average absolute error. Less sensitive to outliers than MSE/RMSE. Smaller is better.

129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - loss: 8695256064.0000
129/129 ━━━━━━━━━━━━━━━━━━━━ 0s 831us/step
Test Set Loss (MSE): 8695256064.0
Mean Squared Error (MSE): 8695256064.0
Root Mean Squared Error (RMSE): 93248.35689705207
Mean Absolute Error (MAE): 67101.0859375


In [10]:
#HDF5 format (legacy, but widely used)
model.save("my_regression_model.keras")

In [11]:
loaded_model = keras.models.load_model("my_regression_model.keras")

sample_data = X_test_scaled[:5]
sample_predictions = loaded_model.predict(sample_data)

print(f"True Values (First 5): {y_test[:5]}")
print(f"Predicted Values (First 5): {sample_predictions.flatten()}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step
True Values (First 5): [ 58800. 165800. 139400. 107500. 107200.]
Predicted Values (First 5): [ 97129.32  115886.125 161868.94  170436.27  122852.64 ]


In [12]:
sample_data

array([[-3.4425768e-01,  6.9964582e-01,  3.4131539e-01, -6.3508970e-01,
        -6.7685544e-01, -6.8900812e-01, -7.0545286e-01, -9.7147924e-01],
       [ 8.2905251e-01, -8.3061808e-01, -2.1506566e-01, -1.4815816e-01,
         8.8674426e-02,  5.1544741e-04,  8.2324520e-02, -7.3807955e-01],
       [ 6.0437745e-01, -7.4638325e-01,  1.5335605e+00, -3.9093292e-01,
        -2.4863718e-01, -3.7885734e-01, -2.3803829e-01, -8.1762624e-01],
       [ 7.0922607e-01, -4.4220346e-01,  2.6183239e-01, -6.0698867e-01,
        -7.1513194e-01, -6.3147289e-01, -6.4768255e-01, -3.3101869e-01],
       [-1.9946566e-01,  1.5419924e+00, -3.7403166e-01, -8.1152755e-01,
        -8.3235371e-01, -9.2274487e-01, -1.0179379e+00, -3.6738589e-01]],
      dtype=float32)